In [16]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [17]:
words = open("files/names.txt",'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [18]:
len(words)

32033

In [19]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [20]:
block_size = 3 # context lenght how many chars we take to predict the next one

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context = [0] * block_size

        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [21]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s,dt,t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt,t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

In [39]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the nuber of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size,n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size,n_hidden),generator=g) * (5/3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden,                      generator=g) * 0.1 # using b1 just for fun, it's useless because of batchnorm
# Layer 2 
W2 = torch.randn((n_hidden,vocab_size),         generator=g) * 0.1
b2 = torch.randn(vocab_size,                    generator=g) * 0.1

# BatchNorm parameters
bngain = torch.ones((1,n_hidden)) * 0.1 + 1.0
bnbias = torch.zeros((1,n_hidden))* 0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g, all zeros could mask an incorrect 
# implementation of the backward pass.


parameters = [C,W1,b1,W2,b2,bngain,bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total 
for p in parameters:
    p.requires_grad = True

12297


In [40]:
batch_size = 32
n = batch_size # a shorte variable also, for convenience
# construct a minibatch 
ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator=g)
Xb,Yb = Xtr[ix],Ytr[ix] # batch X,Y

In [41]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time 

emb = C[Xb] # embed the charcters into vector
embcat = emb.view(emb.shape[0],-1) # concatenate the vectors
# Linear Layer 1 
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm Layer
bnmeani = 1/n*hprebn.sum(0,keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0,keepdim=True)# note: Bessel's correction (dividing by n-1 not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-Linearity
h = torch.tanh(hpreact) # hidden layer
# Linear Layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits,Yb))
logits_maxes = logits.max(1,keepdim=True).values
norm_logits = logits - logits_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1,keepdim=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n),Yb].mean()

for p in parameters:
    p.grad = None

for t in [logprobs,probs,counts,counts_sum,counts_sum_inv,norm_logits,logits_maxes,logits,h,hpreact,bnraw,bnvar_inv,bnvar,bndiff2,bndiff,hprebn,bnmeani,embcat,emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.8279, grad_fn=<NegBackward0>)

dloss/dloss = 1
loss = -logprobs(of n correct probs) = -log(Piyi)
dloss/dlogprobs = -1/n

logprobs = log(probs)
dlogprobs/dprobs = 1/probs
dloss/dprobs = dloss/dlogprobs * dlogprobs/dprobs 

dloss/dcount_sum_inv = dloss/dprobs * dprobs/dcount_sum_inv: dprobs / count_sum_inv

In [42]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n),Yb] = -1.0/n
cmp('logprobs',dlogprobs,logprobs)
dprobs = dlogprobs / probs
cmp("probs",dprobs,probs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [43]:
dcount_sum_inv = dprobs / counts_sum_inv

cmp("counts_sum_inv",dcount_sum_inv,counts_sum_inv)

counts_sum_inv  | exact: False | approximate: False | maxdiff: 42.12601089477539


In [45]:
from torchviz import make_dot
make_dot(loss, params={
    "C": C,
    "W1": W1,
    "b1": b1,
    "W2": W2,
    "b2": b2,
    "bngain": bngain,
    "bnbias": bnbias
}).render("model_graph", format="svg")

'model_graph.svg'